# Object Detection with YOLO11
## Fine-Tuning a Pretrained Detector and Reading Its Metrics

Simone Bonato & Susanne Suter, FHNW

Object detection means finding each object in an image and marking it with a class label and a bounding box. Here, we fine-tune a pretrained YOLO11 model on a small, unusual dataset, inspect its mistakes, and learn to read the metrics detection papers report.

It complements *Know Your Images*, where you explored a detection dataset. Here we train.

*Andrej Karpathy's [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/) applies here too, though we do not follow it step by step.*

**A note about this notebook**

This notebook was developed with AI assistance and reviewed, edited, and extended by the authors. Not every detail could be thoroughly verified.

If you find errors, unexpected behavior, or anything confusing, please let us know.

In [ ]:
# @title Learning Objectives
from IPython.display import HTML

HTML(r"""
<div class="callout callout-banner"
     style="background:#E5E7EB;color:#374151;border-left:6px solid #6B7280;
            border-radius:6px;padding:12px 14px;line-height:1.5;
            font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;
            font-size:14px;width:100%;box-sizing:border-box;
            white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-size:22px;font-weight:bold;line-height:1.3;">Learning Objectives: Object Detection with YOLO11</div>
<p style="margin:6px 0 0 0;"><strong>At the end of this notebook you will be able to:</strong></p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">Fine-tune a pretrained YOLO model on a custom dataset and explain the main training settings.</li>
<li style="margin:2px 0;">Classify detections as true positives, false positives and false negatives, and explain how the
confidence and NMS thresholds change them.</li>
<li style="margin:2px 0;">Interpret a precision–recall curve, AP, mAP@50 and mAP@50–95, and compare two model sizes with them.</li>
</ol>
<p style="margin:8px 0 0 0;">Tasks are marked beginner 🟢, intermediate 🟡, advanced 🔴 (optional).</p>
</div>
""")

**What Do You Need to Know for the Exam and Your Project?**

Focus on concepts and decisions: what fine-tuning changes, what makes a detection a true or false positive, how the two thresholds trade precision against recall, and what each metric rewards. You do **not** need to memorize this dataset's numbers or Ultralytics arguments.

In [ ]:
# @title How to Use This Notebook
from IPython.display import HTML

HTML(r"""
<div class="callout" style="background:#F5F5F5;color:#374151;border-left:6px solid #9CA3AF;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">How to Use This Notebook</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">Make a copy (File → Save a copy in Drive), then select a GPU: Runtime → Change runtime type → T4 GPU.</li>
<li style="margin:2px 0;">Run the cells in order. Training the default model takes a few minutes on a T4; keep reading while it runs.</li>
<li style="margin:2px 0;">Answer each task before you open its 💡 Sample Answer.</li>
<li style="margin:2px 0;">The 🔴 experiments at the end are optional and train two more models.</li>
</ul>
</div>
""")

### Setup

The next cell installs Ultralytics (the library behind YOLO11), checks for a GPU, and loads the helpers. It takes about a minute.

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
# The common imports are made here, once (later cells use these names).
import base64
import os
import random
import sys
import time
from io import BytesIO
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install` (N-021)
    pass
    %pip install -q ultralytics==8.4.208

import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, clear_output, display
from IPython.display import Image as IPythonImage
from PIL import Image
from ultralytics import YOLO

random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():  # Colab GPU (0 = the first GPU)
    DEVICE = 0
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():  # a Mac's own GPU, for running the notebook on a Mac
    DEVICE = "mps"
    print("Apple GPU (MPS) found: training runs on it.")
else:
    DEVICE = "cpu"
    print("No GPU found. Training will be very slow: Runtime → Change runtime type → T4 GPU, then run again.")

RUNS_DIR = Path("runs")  # Ultralytics writes weights and plots here

# --- Theme (SPEC L-004): one meaning per color in every figure ---
TP_COLOR, FP_COLOR, FN_COLOR = "#2E7D32", "#EF6C00", "#C62828"  # found · invented · missed
GT_COLOR = "#FFFFFF"  # ground truth: white dashed outline
INK = "#21434A"
mpl.rcParams.update({"figure.facecolor": "white", "axes.edgecolor": "#B0B0B0", "axes.grid": True,
                     "grid.color": "#E5E7EB", "axes.titlecolor": INK, "axes.labelcolor": INK})


def callout(kind: str, title: str, body_html: str) -> HTML:
    """One box as HTML. kind: "answer" (💡 a gray toggle, closed until opened), "theory" (📘 blue),
    "recipe" (🧾 yellow) or "deeper" (🔍 gray)."""
    background, ink, bar = {
        "answer": ("#F5F5F5", "#374151", "#9CA3AF"),
        "theory": ("#E3F2FD", "#0D47A1", "#0D47A1"),
        "recipe": ("#FFF8D6", "#333333", "#F5C400"),
        "deeper": ("#F3F4F6", "#4B5563", "#4B5563"),
    }[kind]
    style = (f"background:{background};color:{ink};border-left:6px solid {bar};border-radius:6px;"
             "padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;"
             "font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;"
             "margin:2px 0;")
    if kind == "answer":
        inner = (f'<details style="margin:0;">\n<summary style="cursor:pointer;font-weight:bold;">{title}</summary>'
                 f"\n{body_html}\n</details>")
    else:
        inner = f'<div style="font-weight:bold;">{title}</div>\n{body_html}'
    return HTML(f'<div class="callout callout-{kind}" style="{style}">\n{inner}\n</div>')


def show(fig) -> None:
    """Draw a figure as a PNG in the output (the same in Colab, Jupyter and the automated tests)."""
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    display(IPythonImage(data=buf.getvalue()))


def show_live(fig) -> None:
    """Draw a figure inside a slider or Previous / Next view (Colab renders it as an image)."""
    display(fig)
    plt.close(fig)


def fig_to_html(fig, width: int = 100) -> str:
    """A matplotlib figure as an inline <img> (used inside Sample Answers)."""
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    png = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f'<img src="data:image/png;base64,{png}" style="width:{width}%;height:auto;margin:8px 0;">'


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · device: {DEVICE}")

## YOLO: A One-Stage Detector

Learn what a YOLO model predicts for one image and which model size we train.

YOLO ("You Only Look Once") is a **one-stage detector**: one convolutional network looks at the whole image once and outputs boxes, class scores and a confidence per box. Two-stage detectors such as Faster R-CNN first propose regions, then classify each, which is slower.

We use **YOLO11** through the Ultralytics Python API, where training, validation and prediction are each one line.

In [ ]:
# @title 📘 What a YOLO Model Outputs
callout("theory", "📘 What a YOLO Model Outputs", r"""
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Many candidate boxes.</strong> For every location of a coarse grid over the image (at three scales),
the network predicts a box, class scores and a confidence. A 640×640 input gives 8,400 candidates.</li>
<li><strong>A confidence threshold</strong> drops candidates the model is unsure about.</li>
<li><strong>Non-maximum suppression (NMS)</strong> removes duplicates: of several boxes on the same object, only
the most confident is kept.</li>
</ol>
<p style="margin:6px 0 0 0;">You will move both thresholds with a slider later.</p>
""")

<callout: theory>

YOLO11 comes in five sizes: larger models are more accurate but need more memory and time. These are Ultralytics' COCO results at 640 px (mAP@50–95 is explained later).

| Model | Parameters | Compute (GFLOPs) | COCO mAP@50–95 | Typical target |
|---|---|---|---|---|
| YOLO11n | 2.6 M | 6.5 | 39.5 | phones, edge devices |
| YOLO11s | 9.4 M | 21.5 | 47.0 | small GPUs |
| YOLO11m | 20.1 M | 68.0 | 51.5 | consumer GPUs |
| YOLO11l | 25.3 M | 86.9 | 53.4 | cloud GPUs |
| YOLO11x | 56.9 M | 194.9 | 54.7 | high-end cloud GPUs |

We fine-tune **YOLO11n**, the smallest, so training takes minutes. The optional experiments compare it with YOLO11s.

In [ ]:
# @title 🔍 Go Deeper: Why YOLO11 and Not the Newest YOLO?
callout("deeper", "🔍 Go Deeper: Why YOLO11 and Not the Newest YOLO?", r"""
<p style="margin:6px 0 0 0;">Ultralytics' newest family, YOLO26 (2026), can also run <em>NMS-free</em>: a second
output head is trained to give one box per object, so the duplicate-removal step can be skipped at inference time
(<code>nms=False</code>). On small fine-tuning datasets like ours, YOLO11 performs about as well, and its classic
pipeline shows every step you explore in this notebook. Swapping in YOLO26 is a one-word change:
<code>YOLO("yolo26n.pt")</code>.</p>
""")

<callout: deeper>

## The Dataset

Download the dataset, recall YOLO's label format, and look at some images and boxes.

A drone flies low over a field and takes photos. A farmer who knows where the weeds are can spray only those spots, saving herbicide. A detector must find every weed and tell it from the crop.

Our dataset has photos taken from above, with boxes around crop plants and weeds: two classes, almost all weeds. The objects are tiny, around 20 pixels wide in a 640 × 640 image, and look nothing like COCO's photos. We only look briefly here.

The next cell downloads the images (about 55 MB, one to two minutes the first time; later runs reuse them). To keep training short, we use a fixed random subset: 500 training and 150 validation images.

In [ ]:
# @title Download the Dataset { display-mode: "form" }
import shutil

import yaml
from huggingface_hub import list_repo_files, snapshot_download
from ultralytics.data.utils import check_det_dataset

REPO_ID = "LibreYOLO/weed-crop-aerial"  # Roboflow 100 "weed-crop-aerial" in YOLO format, CC BY 4.0, no login needed
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}


def images_in(folder) -> list[Path]:
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)


def label_path_for(image: Path) -> Path:
    """YOLO's convention: the label file sits in a 'labels' folder parallel to 'images', same file name."""
    parts = list(Path(image).parts)
    i = len(parts) - 1 - parts[::-1].index("images")
    return Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")


def download_subset(n_train: int, n_val: int) -> Path:
    """Download a fixed random sample of images and labels, and return the path of its data.yaml.

    Safe to run again: a finished download is reused, an unfinished one is restarted (N-019).
    """
    root = Path(f"data/datasets/weed-crop-aerial-{n_train}-{n_val}").resolve()
    yaml_path = root / "data.yaml"
    if yaml_path.exists():  # written last, so its presence means the download finished
        return yaml_path
    shutil.rmtree(root, ignore_errors=True)
    files = list_repo_files(REPO_ID, repo_type="dataset")
    chosen = []
    for folder, n in (("train", n_train), ("valid", n_val)):
        names = sorted(f for f in files if f.startswith(f"{folder}/images/"))
        for image in random.Random(SEED).sample(names, n):
            chosen += [image, str(Path(image.replace("/images/", "/labels/")).with_suffix(".txt"))]
    snapshot_download(REPO_ID, repo_type="dataset", local_dir=root, allow_patterns=chosen, token=False)
    yaml_path.write_text(yaml.safe_dump({"path": str(root), "train": "train/images", "val": "valid/images",
                                         "names": dict(enumerate(["crop", "weed"]))}))
    return yaml_path


# In smoke mode (automated tests only) the same code downloads a handful of images instead.
data_info = check_det_dataset(str(download_subset(8, 4) if SMOKE else download_subset(500, 150)))
CLASS_NAMES = [data_info["names"][k] for k in sorted(data_info["names"])]
DATA_YAML = Path(data_info["yaml_file"])
TRAIN_IMAGES = images_in(data_info["train"])
VAL_IMAGES = images_in(data_info["val"])
print(f"{len(TRAIN_IMAGES)} training images · {len(VAL_IMAGES)} validation images · classes: {', '.join(CLASS_NAMES)}")

### Data Format Recap

YOLO expects one text file per image, with the same name, in a `labels` folder next to `images`. Each line is one object: class index and box, **normalized** by image width and height (0 to 1):

```
class_idx x_center y_center width height
```

A file `data.yaml` tells Ultralytics where the images are and what the classes are called:

```
path: /content/data/datasets/weed-crop-aerial-500-150
train: train/images
val: valid/images
names:
  0: crop
  1: weed
```

The next cell prints the first lines of one label file and its image size.

In [ ]:
# @title 🔎 Read One Label File { display-mode: "form" }
example_image = TRAIN_IMAGES[0]
with Image.open(example_image) as im:
    print(f"{example_image.name}: {im.width} × {im.height} pixels")
print(f"{label_path_for(example_image).name}:")
print("\n".join(label_path_for(example_image).read_text().splitlines()[:5]))

In [ ]:
# @title Helpers: Ground-Truth Boxes { display-mode: "form" }
def read_gt(image) -> dict:
    """The ground-truth boxes of one image: {"xyxy": N×4 pixel corners, "cls": N class indices}."""
    label = label_path_for(Path(image))
    rows = np.loadtxt(label, ndmin=2) if label.exists() and label.stat().st_size else np.zeros((0, 5))
    with Image.open(image) as im:
        w, h = im.size
    cx, cy, bw, bh = rows[:, 1] * w, rows[:, 2] * h, rows[:, 3] * w, rows[:, 4] * h
    xyxy = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], axis=1)
    return {"xyxy": xyxy, "cls": rows[:, 0].astype(int)}


CLASS_COLORS = ["#E07A5F", "#3D405B", "#81B29A", "#9C89B8", "#F2CC8F", "#2A9D8F"]  # one per class, in order


def show_gt(ax, image, title=""):
    gt = read_gt(image)
    ax.imshow(Image.open(image).convert("RGB"))
    ax.axis("off")
    for box, k in zip(gt["xyxy"], gt["cls"], strict=True):
        x0, y0, x1, y1 = box
        ax.add_patch(mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.8,
                                           edgecolor=CLASS_COLORS[k % len(CLASS_COLORS)]))
    ax.set_title(f"{title}{len(gt['cls'])} boxes", fontsize=9)

### A Short Look at the Training Images

Each page shows four training images with their ground-truth boxes, one color per class. Browse a few pages to see the objects' size, how many appear together, and how they differ from everyday photos.

In [ ]:
# @title 🔎 Browse Training Images with Their Boxes { display-mode: "form" }
gt_page = 0
gt_output = widgets.Output()
gt_prev, gt_next = widgets.Button(description="← Previous"), widgets.Button(description="Next →")
gt_counter = widgets.HTML()
GT_PAGE_SIZE = 4


def gt_show():
    with gt_output:
        clear_output(wait=True)
        page = TRAIN_IMAGES[gt_page * GT_PAGE_SIZE:(gt_page + 1) * GT_PAGE_SIZE]
        fig, axes = plt.subplots(1, GT_PAGE_SIZE, figsize=(16, 4), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, image in zip(axes.ravel(), page, strict=False):
            show_gt(ax, image)
        handles = [mpl.patches.Patch(edgecolor=CLASS_COLORS[k % len(CLASS_COLORS)], fill=False, lw=2, label=name)
                   for k, name in enumerate(CLASS_NAMES)]
        fig.legend(handles=handles, loc="lower center", ncol=len(CLASS_NAMES), frameon=False,
                   bbox_to_anchor=(0.5, -0.06))
        show_live(fig)
    n_pages = -(-len(TRAIN_IMAGES) // GT_PAGE_SIZE)
    gt_counter.value = f"Page {gt_page + 1} of {n_pages}"
    gt_prev.disabled, gt_next.disabled = gt_page == 0, gt_page >= n_pages - 1


def gt_step(step):
    global gt_page
    gt_page += step
    gt_show()


gt_prev.on_click(lambda _: gt_step(-1))
gt_next.on_click(lambda _: gt_step(1))
display(widgets.HBox([gt_prev, gt_next, gt_counter]), gt_output)
gt_show()

### 🟢 Before Fine-Tuning: What Does a COCO Model See?

The pretrained YOLO11n knows COCO's 80 everyday classes: people, cars, dogs, cups. None is one of ours.

**Before running the code:**
- What will it predict on our images: nothing, wrong classes, or our objects under another name?

**Explore:**
- Run the cell to see its predictions on four validation images.

**Observe and interpret:**
- Which COCO classes does it use, if any? Why do we need to fine-tune?

In [ ]:
# @title 🔎 Predict with the COCO-Pretrained Model { display-mode: "form" }
coco_model = YOLO("yolo11n.pt")  # downloads the COCO weights on first use
fig, axes = plt.subplots(1, 4, figsize=(16, 4), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, image in zip(axes.ravel(), VAL_IMAGES[:4], strict=False):
    r = coco_model.predict(str(image), conf=0.25, device="cpu" if SMOKE else DEVICE, verbose=False)[0]
    ax.imshow(r.plot(line_width=2)[..., ::-1])  # r.plot() draws the COCO class names; BGR → RGB
    found = sorted({coco_model.names[int(k)] for k in r.boxes.cls})
    ax.set_title(f"COCO model: {', '.join(found) if found else 'nothing found'}", fontsize=9)
fig.suptitle("Predictions of the COCO-pretrained YOLO11n before fine-tuning (confidence ≥ 0.25)", color=INK)
show(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>It predicts almost nothing.</strong> On the four images the COCO model finds no box at all. Over all
150 validation images it finds a box in only about one image in ten, with the wrong kind of label: a "bird", a
"frisbee" or a "potted plant" on a patch of soil or a leaf. None of COCO's 80 classes is a weed or a crop plant, and
the objects are far smaller than the people and cars it knows.</p>
<p><strong>Why fine-tune:</strong> a detector can only output the classes it was trained on. Its features (edges,
textures, shapes) are still useful, so we keep them and teach the last layers our classes.</p>
""")

<callout: answer>

## Fine-Tune YOLO11n

Choose training settings, fine-tune YOLO11n on our dataset, and read its training curves.

We do not train from scratch. The model starts from COCO weights, so it already knows edges, textures and shapes; fine-tuning adapts it to our classes. A few hundred images and minutes suffice.

In [ ]:
# @title 📘 The Main Training Settings
callout("theory", "📘 The Main Training Settings", r"""
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>epochs</strong>: how many times the model sees every training image. More epochs learn more, until it
memorizes the training set.</li>
<li><strong>batch</strong>: images processed together in one step. Larger batches give smoother gradients but
need more GPU memory.</li>
<li><strong>imgsz</strong>: the pixel size of an image's longer side. Small objects need a larger size.</li>
<li><strong>Data augmentation</strong> is on by default (for example <em>mosaic</em>, which stitches four images
into one), so the model rarely sees the same picture twice.</li>
</ul>
""")

<callout: theory>

### 🟢 Training Settings: How Long and How Big?

YOLO was trained for hundreds of epochs on COCO's 118,000 images. We have a few hundred images, a pretrained model, and a free Colab GPU.

**Before running the code:**
- Will you need more or fewer epochs than for COCO? Why?

**Explore:**
1. Set `EPOCHS` (5 to 50) and `BATCH_SIZE` (8, 16 or 32; a T4 handles 16) in the cell below. Each extra 10 epochs costs about three minutes on a T4.
2. Run the training cell after it.

**Observe and interpret:**
- Did the model stop improving before the last epoch, or would more epochs help?

In [ ]:
# ✏️ YOUR TURN: training settings
EPOCHS = 20  # @param {type:"slider", min:5, max:50, step:5}
BATCH_SIZE = 16  # @param [8, 16, 32] {type:"raw"}
IMG_SIZE = 640  # pixels, the size YOLO11 was pretrained at

if BATCH_SIZE not in (8, 16, 32) or not 1 <= EPOCHS <= 100:
    raise ValueError("Choose BATCH_SIZE from 8, 16, 32 and EPOCHS between 1 and 100.")
print(f"Training for {EPOCHS} epochs with batches of {BATCH_SIZE} images at {IMG_SIZE} px.")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>Far fewer epochs than for COCO.</strong> The model starts from pretrained weights and has only 500
training images, so 20 epochs take a few minutes and already give a usable detector. For these tiny objects the
model is still improving slowly at epoch 20, so 30 epochs may add a little mAP, at the price of about three more
minutes on a T4.</p>
<p><strong>Batch size 16</strong> fits comfortably in a T4's memory for YOLO11n at 640 px. Smaller batches also
work but each epoch takes longer; 32 may run out of memory for the larger YOLO11s.</p>
""")

<callout: answer>

### Train

The next cell fine-tunes YOLO11n and prints one line per epoch: training losses, then precision, recall and mAP on the validation images. **The first epoch is the slowest**, because the images are cached. On a T4 the default run takes 6 to 8 minutes.

In [ ]:
# @title Train YOLO11n { display-mode: "form" }
# In smoke mode (automated tests only) the same call runs with tiny settings on the CPU.
train_settings = dict(
    data=str(DATA_YAML),
    epochs=1 if SMOKE else EPOCHS,
    batch=4 if SMOKE else BATCH_SIZE,
    imgsz=160 if SMOKE else IMG_SIZE,
    device="cpu" if SMOKE else DEVICE,
    workers=0 if SMOKE else 2,
    seed=SEED,
    deterministic=True,
    project=str(RUNS_DIR.resolve()),
    exist_ok=True,  # re-running overwrites this run instead of creating train2, train3, ...
    plots=not SMOKE,
)

model = YOLO("yolo11n.pt")  # COCO-pretrained weights, downloaded on first use
start = time.time()
model.train(name="yolo11n", **train_settings)
train_minutes_n = (time.time() - start) / 60
best_weights = Path(model.trainer.best)  # the epoch with the best validation fitness
print(f"\nTraining took {train_minutes_n:.1f} min. Best weights: {best_weights}")

### 🟢 Training Curves: Is the Model Still Learning?

The left plot shows the box loss (how far predicted boxes are from true ones) per epoch. The right plot shows the validation mAP, a score from 0 to 1 explained later: higher is better.

**Observe and interpret:**
- Do both losses still go down at the last epoch?
- Is the validation mAP still rising or leveled off?
- Would you train longer, shorter, or keep your setting?

In [ ]:
# @title 🔎 Plot the Training Curves { display-mode: "form" }
history = pd.read_csv(Path(model.trainer.save_dir) / "results.csv")
history.columns = history.columns.str.strip()

fig, (ax_loss, ax_map) = plt.subplots(1, 2, figsize=(11, 3.6))
ax_loss.plot(history["epoch"], history["train/box_loss"], "o-", color=INK, label="training images")
ax_loss.plot(history["epoch"], history["val/box_loss"], "o--", color="#E07A5F", label="validation images")
ax_loss.set(title="Box loss (lower is better)", xlabel="epoch", ylabel="loss")
ax_loss.legend()
ax_map.plot(history["epoch"], history["metrics/mAP50(B)"], "o-", color=TP_COLOR, label="mAP@50")
ax_map.plot(history["epoch"], history["metrics/mAP50-95(B)"], "o--", color="#3D405B", label="mAP@50–95")
ax_map.set(title="Validation mAP (higher is better)", xlabel="epoch", ylabel="mAP", ylim=(0, 1))
ax_map.legend()
fig.tight_layout()
show(fig)

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>Both losses fall steadily and are still falling at the end.</strong> The training and validation loss
move together, with no gap that grows, so the model is not overfitting yet.</p>
<p><strong>The validation mAP is noisy at first</strong> and can jump up or down by 0.1 or more between epochs,
because the validation set is small. It settles in the last five epochs at roughly 0.7–0.8 (mAP@50) and 0.4–0.5 (mAP@50–95)
and is still creeping up: 30 epochs would probably add a few points, but not change the picture.</p>
<p><strong>A warning sign would be</strong> a training loss that keeps falling while the validation loss rises:
the model then memorizes the training images (overfitting), and fewer epochs or more augmentation would help.</p>
""")

<callout: answer>

In [ ]:
# @title 🔍 Go Deeper: More Training Plots and Experiment Tracking
callout("deeper", "🔍 Go Deeper: More Training Plots and Experiment Tracking", r"""
<p style="margin:6px 0 0 0;">Ultralytics saves more plots in the run folder (<code>runs/yolo11n/</code> in the
Colab file browser): all losses, example training batches with their augmentations, and a confusion matrix.
For comparing many runs, it can log to tools such as Weights &amp; Biases or MLflow; see the
<a href="https://docs.ultralytics.com/integrations/">Ultralytics integrations</a>. To keep your weights after the
Colab session ends, copy <code>best.pt</code> to your Google Drive.</p>
""")

<callout: deeper>

## Predictions: TP, FP and FN

Predict on the validation images and classify every box as a true positive, false positive or false negative.

A metric is only a summary. Before trusting one, look at the predictions: what the model finds, invents and misses.

In [ ]:
# @title 📘 True Positives, False Positives, False Negatives
callout("theory", "📘 True Positives, False Positives, False Negatives", r"""
<p style="margin:6px 0 0 0;">A predicted box is compared with the ground-truth boxes of the same class using the
<strong>intersection over union (IoU)</strong>: the overlap area divided by the area the two boxes cover together
(0 = no overlap, 1 = identical).</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><span style="color:#2E7D32;font-weight:bold;">True positive (TP)</span>: a prediction that overlaps a
ground-truth box of the same class with IoU ≥ 0.5. Each ground-truth box is matched once, by the most
confident prediction.</li>
<li><span style="color:#EF6C00;font-weight:bold;">False positive (FP)</span>: a prediction with no such match: a
duplicate, wrong class, badly placed box, or nonexistent object.</li>
<li><span style="color:#C62828;font-weight:bold;">False negative (FN)</span>: a ground-truth box no
prediction matched: a missed object.</li>
</ul>
<p style="margin:6px 0 0 0;">Detection has no true negatives: "background" is not countable.</p>
""")

<callout: theory>

In [ ]:
# @title Helpers: Matching and Drawing { display-mode: "form" }
MATCH_IOU = 0.5  # IoU a prediction needs to count as a true positive


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """IoU of every box in a (N×4) with every box in b (M×4), boxes as x0, y0, x1, y1 in pixels."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x0 = np.maximum(a[:, None, 0], b[None, :, 0])
    y0 = np.maximum(a[:, None, 1], b[None, :, 1])
    x1 = np.minimum(a[:, None, 2], b[None, :, 2])
    y1 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter)


def match(pred: dict, gt: dict, iou_thr: float = MATCH_IOU) -> tuple[np.ndarray, np.ndarray]:
    """Greedy matching, most confident prediction first. Returns (is_tp per prediction, is_found per GT box)."""
    is_tp = np.zeros(len(pred["conf"]), bool)
    found = np.zeros(len(gt["cls"]), bool)
    ious = box_iou(pred["xyxy"], gt["xyxy"])
    for i in np.argsort(-pred["conf"]):
        candidates = np.where((gt["cls"] == pred["cls"][i]) & ~found & (ious[i] >= iou_thr))[0]
        if len(candidates):
            best = candidates[np.argmax(ious[i, candidates])]
            found[best] = is_tp[i] = True
    return is_tp, found


def predict_all(weights, images, conf: float, iou: float = 0.7) -> dict:
    """Predictions for every image path: {path: {"xyxy", "conf", "cls"}} as numpy arrays."""
    detector = YOLO(str(weights))
    out = {}
    for path in images:
        r = detector.predict(str(path), conf=conf, iou=iou, imgsz=160 if SMOKE else IMG_SIZE,
                             device="cpu" if SMOKE else DEVICE, verbose=False)[0]
        out[path] = {"xyxy": r.boxes.xyxy.cpu().numpy(), "conf": r.boxes.conf.cpu().numpy(),
                     "cls": r.boxes.cls.cpu().numpy().astype(int)}
    return out


def keep_above(pred: dict, conf: float) -> dict:
    """The predictions with confidence ≥ conf."""
    keep = pred["conf"] >= conf
    return {k: v[keep] for k, v in pred.items()}


def outline(ax, box, color, dashed=False, label=None):
    """One box on an image axis; a thin dark edge keeps light colors visible on bright images."""
    x0, y0, x1, y1 = box
    rect = mpl.patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.8, edgecolor=color,
                                 linestyle="--" if dashed else "-")
    rect.set_path_effects([path_effects.Stroke(linewidth=3, foreground="#00000088"), path_effects.Normal()])
    ax.add_patch(rect)
    if label:
        ax.text(x0, y0 - 2, label, color="white", fontsize=7, va="bottom",
                bbox=dict(facecolor=color, edgecolor="none", pad=1))


def draw_matches(ax, path, pred: dict, title: str = "", show_conf: bool = True) -> tuple[int, int, int]:
    """Image with GT (white dashed), TP (green), FP (orange) and missed GT (red). Returns (TP, FP, FN)."""
    gt = read_gt(path)
    is_tp, found = match(pred, gt)
    ax.imshow(Image.open(path).convert("RGB"))
    ax.axis("off")
    for box, hit in zip(gt["xyxy"], found, strict=True):
        outline(ax, box, GT_COLOR if hit else FN_COLOR, dashed=True)
    for box, c, k, tp in zip(pred["xyxy"], pred["conf"], pred["cls"], is_tp, strict=True):
        label = f"{CLASS_NAMES[k]} {c:.2f}" if show_conf else None
        outline(ax, box, TP_COLOR if tp else FP_COLOR, label=label)
    counts = int(is_tp.sum()), int((~is_tp).sum()), int((~found).sum())
    ax.set_title(f"{title}TP {counts[0]} · FP {counts[1]} · FN {counts[2]}", fontsize=10)
    return counts


def match_legend(fig):
    """The legend every TP/FP/FN figure carries (N-059)."""
    handles = [mpl.lines.Line2D([], [], color=TP_COLOR, lw=2, label="true positive (found)"),
               mpl.lines.Line2D([], [], color=FP_COLOR, lw=2, label="false positive (invented)"),
               mpl.lines.Line2D([], [], color=FN_COLOR, lw=2, ls="--", label="false negative (missed)"),
               mpl.lines.Line2D([], [], color="#9CA3AF", lw=2, ls="--", label="ground truth, matched (white)")]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.04))

### Predict on the Validation Images

The next cell runs the fine-tuned model on every validation image and keeps every box with confidence of at least `SHOW_CONF` = 0.25, the Ultralytics default. Later sections lower it.

In [ ]:
# @title Predict on the Validation Images { display-mode: "form" }
SHOW_CONF = 0.25
val_predictions = predict_all(best_weights, VAL_IMAGES, conf=SHOW_CONF)
n_boxes = sum(len(p["conf"]) for p in val_predictions.values())
n_gt = sum(len(read_gt(p)["cls"]) for p in VAL_IMAGES)
print(f"{len(VAL_IMAGES)} validation images: {n_boxes} predicted boxes, {n_gt} ground-truth boxes.")

### 🟢 True Positives, False Positives, False Negatives: What Went Wrong?

Each image shows the ground truth as dashed outlines and the predictions as solid boxes with class and confidence, colored by the rule above.

**Before running the code:**
- Which mistake do you expect more often: invented weeds (FP) or missed weeds (FN)?

**Explore:**
- Browse at least ten validation images (Previous / Next).

**Observe and interpret:**
- Find one example of each: a true positive, a false positive, a false negative.
- Why might the model have made each FP and FN? Consider object size, occlusion, the image edge, and whether the ground truth is right.

In [ ]:
# @title 🔎 Browse Predictions vs. Ground Truth { display-mode: "form" }
browse_index = 0
browse_output = widgets.Output()
browse_prev = widgets.Button(description="← Previous")
browse_next = widgets.Button(description="Next →")
browse_counter = widgets.HTML()


def browse_show():
    with browse_output:
        clear_output(wait=True)
        path = VAL_IMAGES[browse_index]
        fig, ax = plt.subplots(figsize=(8, 6))
        draw_matches(ax, path, val_predictions[path], title=f"{Path(path).name} · ")
        match_legend(fig)
        show_live(fig)
    browse_counter.value = f"Image {browse_index + 1} of {len(VAL_IMAGES)}"
    browse_prev.disabled = browse_index == 0
    browse_next.disabled = browse_index >= len(VAL_IMAGES) - 1


def browse_step(step):
    global browse_index
    browse_index = min(max(browse_index + step, 0), len(VAL_IMAGES) - 1)
    browse_show()


browse_prev.on_click(lambda _: browse_step(-1))
browse_next.on_click(lambda _: browse_step(1))
display(widgets.HBox([browse_prev, browse_next, browse_counter]), browse_output)
browse_show()

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>True positives</strong> are weeds that stand out from bare soil: a small green plant with nothing else
around it. At confidence 0.25 the model finds roughly four out of five of the labeled objects.</p>
<p><strong>False positives are more frequent than false negatives</strong>, about three times as many. Almost all
of them are small spots with no label nearby, not duplicates: tiny seedlings, leaf fragments, straw or dark patches
of soil. Many look like the objects that <em>are</em> labeled, so the labeling is possibly incomplete there and the
model right; only a look at the image can tell.</p>
<p><strong>False negatives</strong> are mostly the smallest objects (about 17 pixels across, against about 21 for
all objects), weeds that look unlike the typical small seedling (for example one much larger than the others), and
weeds in a planter between similar-looking crop seedlings.</p>
""")

<callout: answer>

## Precision, Recall and AP

Explore how the confidence threshold trades precision against recall, and how AP, mAP@50 and mAP@50–95 summarize it.

Every box comes with a confidence. Which boxes we keep depends on a threshold, and each threshold gives different TP, FP and FN counts. Two numbers summarize them:

- **Precision** = TP / (TP + FP): the share of shown boxes that are real objects.
- **Recall** = TP / (TP + FN): the share of real objects the model finds.

The next cell predicts again, keeping every box down to confidence 0.001, as Ultralytics does for metrics. Each threshold below then just filters these boxes.

In [ ]:
# @title Predict with a Very Low Threshold { display-mode: "form" }
METRIC_CONF = 0.001  # Ultralytics' threshold for metrics: keep (almost) everything, then sweep
all_predictions = predict_all(best_weights, VAL_IMAGES, conf=METRIC_CONF)
gt_all = {p: read_gt(p) for p in VAL_IMAGES}
print(f"{sum(len(p['conf']) for p in all_predictions.values())} candidate boxes on {len(VAL_IMAGES)} images.")

In [ ]:
# @title Helpers: Precision–Recall Curves and AP { display-mode: "form" }
trapezoid = getattr(np, "trapezoid", None) or np.trapz  # NumPy 2 renamed trapz


def pr_curves(predictions: dict, iou_thr: float = MATCH_IOU) -> dict:
    """Per class: confidences (high → low) with the precision and recall when keeping boxes down to each one."""
    confs, tps, classes, n_gt = [], [], [], np.zeros(len(CLASS_NAMES), int)
    for path, pred in predictions.items():
        is_tp, _ = match(pred, gt_all[path], iou_thr)
        confs.append(pred["conf"])
        tps.append(is_tp)
        classes.append(pred["cls"])
        n_gt += np.bincount(gt_all[path]["cls"], minlength=len(CLASS_NAMES))
    confs, tps, classes = (np.concatenate(x) if x else np.zeros(0) for x in (confs, tps, classes))
    curves = {}
    for k in range(len(CLASS_NAMES)):
        order = np.argsort(-confs[classes == k])
        tp = tps[classes == k][order].astype(float)
        cum_tp, cum_fp = np.cumsum(tp), np.cumsum(1 - tp)
        curves[k] = {"conf": confs[classes == k][order],
                     "precision": cum_tp / np.maximum(cum_tp + cum_fp, 1),
                     "recall": cum_tp / max(n_gt[k], 1), "n_gt": int(n_gt[k])}
    return curves


def average_precision(curve: dict) -> float:
    """Area under the precision–recall curve, made monotonic and sampled at 101 recall points (COCO style)."""
    recall = np.concatenate(([0.0], curve["recall"], [1.0]))
    precision = np.concatenate(([1.0], curve["precision"], [0.0]))
    precision = np.flip(np.maximum.accumulate(np.flip(precision)))  # the best precision at this recall or more
    x = np.linspace(0, 1, 101)
    return float(trapezoid(np.interp(x, recall, precision), x))


def at_threshold(curve: dict, conf: float) -> tuple[float, float]:
    """(precision, recall) when keeping the boxes with confidence ≥ conf."""
    n = int((curve["conf"] >= conf).sum())
    if n == 0:
        return 1.0, 0.0
    return float(curve["precision"][n - 1]), float(curve["recall"][n - 1])


CLASS_LINE_COLORS = ["#3D405B", "#E07A5F", "#81B29A", "#9C89B8", "#F2CC8F", "#2A9D8F"]
curves_50 = pr_curves(all_predictions)

### 🟡 The Confidence Threshold: Precision or Recall?

The drone sprays every spot with a weed box. An invented box wastes herbicide; a missed weed keeps growing.

**Before running the code:**
- Raising the confidence threshold from 0.1 to 0.8: what happens to precision and recall? Write down your guess.

**Explore:**
1. Move the `confidence` slider from low to high. The left panel shows one image at that threshold; the right shows each class's precision–recall curve, with a dot at the threshold.
2. Repeat on two or three other images with the `image` slider.

**Observe and interpret:**
- What happens to the orange (invented) and the red (missed) boxes as the threshold rises?
- Which threshold would you choose for spraying, and why is none best for both precision and recall?

In [ ]:
# @title 🔎 Explore the Confidence Threshold { display-mode: "form" }
def show_threshold(conf: float, image: int):
    path = VAL_IMAGES[image]
    fig, (ax_img, ax_pr) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.25, 1]})
    draw_matches(ax_img, path, keep_above(all_predictions[path], conf), title=f"confidence ≥ {conf:.2f} · ")
    for k, curve in curves_50.items():
        color = CLASS_LINE_COLORS[k % len(CLASS_LINE_COLORS)]
        ax_pr.plot(curve["recall"], curve["precision"], color=color, lw=1.8, label=CLASS_NAMES[k])
        p, r = at_threshold(curve, conf)
        ax_pr.plot(r, p, "o", color=color, ms=10, mec="black")
        ax_pr.annotate(f"P {p:.2f} · R {r:.2f}", (r, p), textcoords="offset points", xytext=(8, -14),
                       fontsize=9, color=color)
    ax_pr.set(xlim=(0, 1.02), ylim=(0, 1.05), xlabel="recall: share of real objects found",
              ylabel="precision: share of boxes that are real", title="Precision–recall curve per class (IoU ≥ 0.5)")
    ax_pr.legend(loc="lower left", title="class (dot = current threshold)", fontsize=8, title_fontsize=8)
    match_legend(fig)
    show_live(fig)


_ = widgets.interact(show_threshold,
                     conf=widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, continuous_update=False,
                                              description="confidence"),
                     image=widgets.IntSlider(value=0, min=0, max=len(VAL_IMAGES) - 1, continuous_update=False,
                                             description="image"))

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>Raising the threshold increases precision and lowers recall.</strong> Low-confidence boxes are mostly
false positives, so removing them makes the shown boxes more trustworthy (fewer orange boxes), but some real
objects also had low confidence and are now missed (more red boxes). The dot slides left and up along the curve.
Over the validation images, precision rises from roughly 0.5 at a threshold of 0.1 to 0.8–0.9 at 0.5 and above,
while recall falls from roughly 0.9 to 0.5 at 0.5 and almost nothing at 0.8. This model is rarely very confident,
because the objects are tiny, so recall collapses early.</p>
<p><strong>For spraying</strong>, a missed weed is usually worse than a wasted spot, so a low threshold of about
0.2–0.3 is defensible, which keeps recall around 0.85 at a precision of about 0.6–0.65. How low depends on what an
empty spray patch costs against a weed left growing: that is a decision about the farm, not about the model. No
threshold is best for both, because the same low-confidence boxes contain both the invented weeds and some of the
real ones.</p>
""")

<callout: answer>

### AP: The Whole Curve in One Number

Choosing a threshold is a deployment decision. To compare *models*, we want one number that ignores it: the **average precision (AP)**, the area under the precision–recall curve. A perfect detector has AP = 1. **mAP** is the mean AP over all classes. Ultralytics computes it too, as the next cell shows.

In [ ]:
# @title 🔎 Compute AP per Class and Compare with Ultralytics { display-mode: "form" }
val_metrics = YOLO(str(best_weights)).val(data=str(DATA_YAML), imgsz=160 if SMOKE else IMG_SIZE, batch=4 if SMOKE else 16,
                                          device="cpu" if SMOKE else DEVICE, plots=False, verbose=False)
ap_50 = {k: average_precision(c) for k, c in curves_50.items()}

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(4.2 * len(CLASS_NAMES), 3.6), squeeze=False)
for ax, (k, curve) in zip(axes.ravel(), curves_50.items(), strict=True):
    recall = np.concatenate(([0.0], curve["recall"]))
    precision = np.flip(np.maximum.accumulate(np.flip(np.concatenate(([1.0], curve["precision"])))))
    color = CLASS_LINE_COLORS[k % len(CLASS_LINE_COLORS)]
    ax.fill_between(recall, precision, step="post", color=color, alpha=0.25, label=f"area = AP = {ap_50[k]:.2f}")
    ax.plot(curve["recall"], curve["precision"], color=color, lw=1.5, label="precision–recall curve")
    ax.set(xlim=(0, 1.02), ylim=(0, 1.05), xlabel="recall", ylabel="precision",
           title=f"{CLASS_NAMES[k]} ({curve['n_gt']} objects)")
    ax.legend(loc="lower left", fontsize=8)
fig.suptitle("AP is the shaded area under each class's curve (IoU ≥ 0.5)", color=INK)
fig.tight_layout()
show(fig)
print(f"mAP@50, our computation: {np.mean(list(ap_50.values())):.3f} (mean of the class APs above)")
print(f"mAP@50, Ultralytics val:  {val_metrics.box.map50:.3f} (small differences come from how boxes are matched)")

### 🟡 mAP@50 vs. mAP@50–95: What Does Each Reward?

So far a prediction counted as correct at IoU ≥ 0.5, which a box covering two-thirds of an object can pass. **mAP@50–95** averages the AP at ten IoU thresholds, 0.50, 0.55, …, 0.95. It is the main COCO metric, used in the model table at the top.

**Before running the code:**
- Will AP go up or down as the required IoU rises from 0.50 to 0.95?

**Explore:**
- Run the cell. Each bar is the mAP at one IoU threshold; the dashed line is their mean, mAP@50–95.

**Observe and interpret:**
- How large is the gap between mAP@50 and mAP@50–95? What does it tell you about the boxes?
- Name one application where mAP@50 is the number to watch, and one where mAP@50–95 matters more.

In [ ]:
# @title 🔎 AP at Ten IoU Thresholds { display-mode: "form" }
IOU_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)
map_per_iou = [np.mean([average_precision(c) for c in pr_curves(all_predictions, t).values()]) for t in IOU_THRESHOLDS]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar([f"{t:.2f}" for t in IOU_THRESHOLDS], map_per_iou, color=["#2E7D32"] + ["#81B29A"] * 9)
ax.axhline(np.mean(map_per_iou), ls="--", color=INK, label=f"mean = mAP@50–95 = {np.mean(map_per_iou):.2f}")
ax.set(xlabel="IoU a prediction needs to count as a true positive", ylabel="mAP", ylim=(0, 1),
       title=f"mAP@50 = {map_per_iou[0]:.2f} (dark bar) · stricter thresholds reward tighter boxes")
ax.legend()
show(fig)
print(f"mAP@50–95, our computation: {np.mean(map_per_iou):.3f} · Ultralytics val: {val_metrics.box.map:.3f}")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>AP falls as the required IoU rises, and here it falls fast.</strong> mAP@50 is roughly 0.75, but
the bars drop to about half at IoU 0.75 and to almost nothing at 0.9, so mAP@50–95 is only about 0.45. The gap of
about 0.3 means the model <em>finds</em> the weeds, but its boxes are not tight. This is expected for tiny
objects: for a weed 20 pixels wide, a box shifted by just 3 pixels in both directions already has an IoU of only about 0.55.</p>
<p><strong>mAP@50</strong> is the right number when finding is what matters: is there a weed at this spot, so that the
sprayer aims there. <strong>mAP@50–95</strong> matters when the box itself is used: measuring how much of the field
is covered by weeds, or choosing the dose from the size of each weed.</p>
<p><strong>The crop AP is very noisy:</strong> only about 25 of the validation boxes are crop plants, so one
more or fewer found box moves it a lot.</p>
""")

<callout: answer>

## Non-Maximum Suppression

See how the NMS threshold trades duplicate boxes against missed neighbors.

The network proposes thousands of candidate boxes, and a clearly visible object usually gets several overlapping ones. **Non-maximum suppression (NMS)** keeps one box per object:

1. Sort the boxes of one class by confidence.
2. Keep the most confident box.
3. Delete every remaining box whose IoU with the kept box exceeds the **NMS IoU threshold**.
4. Repeat with the next box still left.

The threshold is the `iou` argument of `predict`. We use 0.7, the value Ultralytics uses for metrics (its plain-prediction default is 0.5).

### 🟡 NMS Threshold: Duplicates or Missed Neighbors?

The image below is the validation image with the most annotated objects, about twenty weeds spread over it.

**Before running the code:**
- A *low* NMS threshold deletes even slightly overlapping boxes. What could go wrong with two real objects side by side?
- A threshold of 1.0 deletes nothing. What will you see?

**Explore:**
1. Move the `NMS IoU` slider from 0.1 to 1.0. The left panel shows the default 0.7, the right your setting.
2. Watch the TP, FP and FN counts in the titles.

**Observe and interpret:**
- At which values do duplicates appear (orange boxes on a green one)?
- At which values, if any, do real objects get deleted (red dashed outlines beside a green box)?
- Why does the best value depend on how crowded the scenes are?

In [ ]:
# @title 🔎 Explore the NMS Threshold { display-mode: "form" }
NMS_CONF = 0.25  # the display threshold of section Predictions: TP, FP and FN
crowded_image = max(VAL_IMAGES, key=lambda p: len(read_gt(p)["cls"]))
nms_detector = YOLO(str(best_weights))


def predict_with_nms(nms_iou: float) -> dict:
    r = nms_detector.predict(str(crowded_image), conf=NMS_CONF, iou=nms_iou, imgsz=160 if SMOKE else IMG_SIZE,
                             device="cpu" if SMOKE else DEVICE, verbose=False)[0]
    return {"xyxy": r.boxes.xyxy.cpu().numpy(), "conf": r.boxes.conf.cpu().numpy(),
            "cls": r.boxes.cls.cpu().numpy().astype(int)}


nms_default = predict_with_nms(0.7)


def show_nms(nms_iou: float):
    fig, (ax_default, ax_yours) = plt.subplots(1, 2, figsize=(14, 5.5))
    draw_matches(ax_default, crowded_image, nms_default, title="default NMS IoU 0.70 · ", show_conf=False)
    draw_matches(ax_yours, crowded_image, predict_with_nms(nms_iou), title=f"your NMS IoU {nms_iou:.2f} · ",
                 show_conf=False)
    match_legend(fig)
    show_live(fig)


_ = widgets.interact(show_nms, nms_iou=widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05,
                                                           continuous_update=False, description="NMS IoU"))

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>Close to 1.0, duplicates appear:</strong> almost no box is deleted, so one object is covered by several
boxes; only one of them can be the true positive, the others count as false positives. On this image the false
positives stay at about 4 up to 0.75, then grow to over 10 at 0.85 and to more than 50 at 0.95, while the true
positives do not change.</p>
<p><strong>At low values nothing happens here:</strong> the red outlines and the green boxes stay the same from 0.1
to 0.75. NMS can only delete a real neighbor when two real objects overlap, and the weeds in these images are tiny
and stand apart, so their boxes almost never touch. In a different dataset the low end would show lost neighbors.</p>
<p><strong>The best value depends on crowding:</strong> where objects never overlap, any value from low to about
0.7 is safe, as here; in dense scenes (herds, shelves, crowds, weeds growing in a clump) real objects overlap, and a
higher threshold is needed to keep them.</p>
""")

<callout: answer>

## Optional Experiments

Compare a larger model with ours, and see what freezing the backbone changes.

Each experiment trains another model with the same settings, taking a few minutes on a T4. Tick the box below to run them; unticked, the cells skip themselves and "Run all" stays fast.

In [ ]:
# ✏️ YOUR TURN: run the optional experiments?
RUN_EXPERIMENTS = False  # @param {type:"boolean"}

### 🔴 YOLO11n vs. YOLO11s: Is the Larger Model Worth It? (Optional)

YOLO11s has about 3.6 times as many parameters as YOLO11n.

**Before running the code:**
- How much better do you expect YOLO11s to be on our small dataset, and how much slower?

**Explore:**
- Run the cell. It trains YOLO11s with your settings and compares both models on the validation images.

**Observe and interpret:**
- Compare mAP@50, mAP@50–95, training time and prediction time per image.
- For which use would you choose each model?

In [ ]:
# @title 🔎 Train YOLO11s and Compare { display-mode: "form" }
def evaluate(weights) -> dict:
    """mAP and speed of one trained model on the validation images."""
    detector = YOLO(str(weights))
    m = detector.val(data=str(DATA_YAML), imgsz=160 if SMOKE else IMG_SIZE, batch=4 if SMOKE else 16,
                     device="cpu" if SMOKE else DEVICE, plots=False, verbose=False)
    n_params = sum(p.numel() for p in detector.model.parameters())
    return {"parameters (millions)": round(n_params / 1e6, 1), "mAP@50": round(float(m.box.map50), 3),
            "mAP@50–95": round(float(m.box.map), 3), "prediction ms / image": round(m.speed["inference"], 1)}


if RUN_EXPERIMENTS or SMOKE:  # smoke mode runs the same code with tiny settings
    model_s = YOLO("yolo11s.pt")
    start = time.time()
    model_s.train(name="yolo11s", **train_settings)
    train_minutes_s = (time.time() - start) / 60
    comparison = pd.DataFrame({
        "YOLO11n": {**evaluate(best_weights), "training minutes": round(train_minutes_n, 1)},
        "YOLO11s": {**evaluate(model_s.trainer.best), "training minutes": round(train_minutes_s, 1)},
    })
    display(comparison)
else:
    print("Skipped: tick RUN_EXPERIMENTS above and run this cell again.")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>On this small dataset, YOLO11s is not clearly better.</strong> In our test run both models reached about
the same mAP@50 (roughly 0.75–0.8) and mAP@50–95 (roughly 0.45–0.5), and YOLO11s was sometimes a little lower,
while it took two to three times as long to train and was slower per image.</p>
<p><strong>Why:</strong> with a few hundred images, the data, not the model size, limits the result; the larger
model has more to learn from too little. <strong>Choose YOLO11n</strong> for a drone or a small chip, or when
many images per second are needed. A larger model pays off with more data, or when every missed object counts.
Always compare on the validation images, never on the training images.</p>
""")

<callout: answer>

### 🔴 Freezing the Backbone: Faster, but How Good? (Optional)

The first layers of a YOLO model, the *backbone*, extract general image features. We can **freeze** them so only later layers train. In YOLO11, `freeze=10` freezes the whole backbone.

**Before running the code:**
- Will training be faster or slower? Will the mAP be higher or lower than without freezing?

**Explore:**
- Run the cell. It retrains YOLO11n with the backbone frozen and adds it to the comparison.

**Observe and interpret:**
- When would freezing be a good choice?

In [ ]:
# @title 🔎 Train YOLO11n with a Frozen Backbone { display-mode: "form" }
if RUN_EXPERIMENTS or SMOKE:
    model_frozen = YOLO("yolo11n.pt")
    start = time.time()
    model_frozen.train(name="yolo11n-frozen", freeze=10, **train_settings)
    train_minutes_frozen = (time.time() - start) / 60
    comparison["YOLO11n, frozen backbone"] = {**evaluate(model_frozen.trainer.best),
                                              "training minutes": round(train_minutes_frozen, 1)}
    display(comparison)
else:
    print("Skipped: tick RUN_EXPERIMENTS above and run this cell again.")

In [ ]:
# @title 💡 Sample Answer { display-mode: "form" }
callout("answer", "💡 Sample Answer", r"""
<p><strong>The mAP is lower</strong> (in our run mAP@50 dropped from about 0.78 to about 0.72), because the
features stay those learned for COCO's everyday photos, and aerial images of plants look nothing like them.
<strong>Training is not faster here</strong>, unlike what you might expect: for such a small model the frozen
layers still compute their outputs, only their gradients are skipped, and the saving is lost in the rest of the
work (in our test runs the frozen model even took longer).</p>
<p>Freezing is a good choice with very few training images (it reduces overfitting), with little GPU memory, or
when the new images look like the pretraining images. The further our images are from COCO, the more it helps to
fine-tune the backbone as well.</p>
""")

<callout: answer>

In [ ]:
# @title 📘 What Matters for Object Detection?
callout("theory", "📘 What Matters for Object Detection?", r"""
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li><strong>Start from pretrained weights.</strong> A few hundred images and minutes of fine-tuning give a usable
detector.</li>
<li><strong>Look at the errors, not only the metric.</strong> FPs and FNs show what the model, and sometimes the
labels, get wrong.</li>
<li><strong>Thresholds are decisions.</strong> Confidence trades precision for recall; NMS trades duplicates for
missed neighbors. Choose for the application.</li>
<li><strong>Report AP, not one threshold.</strong> mAP@50 says whether objects are found; mAP@50–95 also how tight
the boxes are.</li>
</ul>
""")

<callout: theory>